# TP 1 - BiLSTM pour l'Analyse de Sentiment

**Seance 4 - Architectures Sequentielles Avancees** - Golden Collar

## Objectifs
1. Implementer un **BiLSTM** avec TensorFlow/Keras
2. L'appliquer a l'**analyse de sentiment** (IMDB)
3. Comparer **LSTM simple** vs **BiLSTM** vs **Stacked BiLSTM** (a tete identique)
4. Maitriser `Embedding`, `return_sequences`, `Bidirectional`, `Dropout`

## Rappel
Le BiLSTM traite la sequence dans les deux directions :
$$h_t = [\overrightarrow{h}_t \;; \; \overleftarrow{h}_t] \in \mathbb{R}^{2d}$$
Exemple : *"Ce film n'est **pas** du tout **mauvais**"* - le contexte bidirectionnel est crucial.

> **Fil conducteur :** un seul `build_model(kind=...)` (meme tete pour les 3 modeles, donc
> comparaison equitable) et un helper `train()`.

---
## Partie 1 - Configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, TensorBoard
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow :", tf.__version__)
print("GPU :", tf.config.list_physical_devices('GPU') or "aucun (CPU)")

---
## Partie 2 - Donnees IMDB

50 000 critiques etiquetees positives/negatives, deja tokenisees en indices entiers.

In [ ]:
VOCAB_SIZE    = 10000   # top 10 000 mots
MAX_LEN       = 200     # longueur des sequences (padding/troncature)
EMBEDDING_DIM = 128     # dimension des vecteurs d'embedding

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
print(f"Train {len(X_train)} | Test {len(X_test)} critiques")
print("Distribution train :", np.bincount(y_train))

In [ ]:
lengths = [len(s) for s in X_train]
plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(MAX_LEN, color='red', ls='--', lw=2, label=f'MAX_LEN = {MAX_LEN}')
plt.xlabel('longueur (mots)'); plt.ylabel('nb critiques'); plt.legend()
plt.title('Longueur des critiques IMDB'); plt.tight_layout(); plt.show()
print(f"mediane {np.median(lengths):.0f} mots | couvert par MAX_LEN : "
      f"{100*np.mean(np.array(lengths) <= MAX_LEN):.1f}%")

In [ ]:
# Padding (pre-padding + troncature des sequences trop longues)
X_train_pad = pad_sequences(X_train, maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test_pad  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='pre', truncating='pre')
print("X_train :", X_train_pad.shape, "| X_test :", X_test_pad.shape)

In [ ]:
# Decodage d'une critique (verification)
word_index = imdb.get_word_index()
rev = {v + 3: k for k, v in word_index.items()}
rev.update({0: '<PAD>', 1: '<START>', 2: '<UNK>', 3: '<UNUSED>'})

def decode(seq):
    return ' '.join(rev.get(i, '?') for i in seq if i != 0)

print(f"Label = {'positif' if y_train[0] else 'negatif'}")
print(decode(X_train[0][:60]), "...")

---
## Partie 3 - Jeu de validation (80/20)

In [ ]:
VAL_SPLIT = 5000
X_val, y_val = X_train_pad[:VAL_SPLIT], y_train[:VAL_SPLIT]
X_tr,  y_tr  = X_train_pad[VAL_SPLIT:], y_train[VAL_SPLIT:]
print(f"Train {len(X_tr)} | Val {len(X_val)} | Test {len(X_test_pad)}")

BATCH_SIZE, EPOCHS, PATIENCE = 32, 15, 5
LOG_DIR = os.path.join("logs", "bilstm_tp1")

---
## Les briques reutilisables

`build_model(kind=...)` : **meme tete** (`Dense(32) -> Dropout -> Dense(1)`) pour les trois
variantes, afin d'isoler l'effet de l'architecture recurrente. `train()` compile + entraine.

In [ ]:
def build_model(kind='bilstm', units=64, vocab=VOCAB_SIZE, emb=EMBEDDING_DIM, max_len=MAX_LEN):
    inputs = Input(shape=(max_len,))
    x = Embedding(vocab, emb)(inputs)
    x = SpatialDropout1D(0.5)(x)
    # recurrent_dropout volontairement non utilise (il desactive le noyau cuDNN -> lent)
    if kind == 'lstm':
        x = LSTM(units, dropout=0.4)(x)
    elif kind == 'bilstm':
        x = Bidirectional(LSTM(units, dropout=0.4))(x)
    elif kind == 'stacked':
        x = Bidirectional(LSTM(units, return_sequences=True, dropout=0.4))(x)  # return_sequences=True obligatoire
        x = Bidirectional(LSTM(units // 2, dropout=0.4))(x)
    else:
        raise ValueError(kind)
    x = Dense(32, activation='relu')(x)     # tete commune
    x = Dropout(0.4)(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inputs, out, name=kind)


def train(model, name):
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=5e-4, weight_decay=2e-4),
                  loss='binary_crossentropy', metrics=['accuracy'])
    cbs = [EarlyStopping('val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1),
           ReduceLROnPlateau('val_loss', factor=0.5, patience=2, verbose=0),
           TensorBoard(log_dir=os.path.join(LOG_DIR, name), histogram_freq=0)]
    return model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                     epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cbs, verbose=1)

---
## Partie 4 - Modele 1 : LSTM simple (baseline)

In [ ]:
model_lstm = build_model('lstm')
model_lstm.summary()

In [ ]:
# Verification du nombre de parametres du LSTM
# Params_LSTM = 4 * [d*(d+n) + d] = 4*d*(d+n+1), avec d=units, n=EMBEDDING_DIM
d, n = 64, EMBEDDING_DIM
manuel = 4 * d * (d + n + 1)
keras_lstm = next(l for l in model_lstm.layers if l.__class__.__name__ == 'LSTM').count_params()
print(f"Manuel : 4 x {d} x ({d}+{n}+1) = {manuel:,}")
print(f"Keras  : {keras_lstm:,}   ->  {'OK' if manuel == keras_lstm else 'ecart'}")

In [ ]:
history_lstm = train(model_lstm, 'lstm_simple')

---
## Partie 5 - Modele 2 : BiLSTM

Forward + backward concatenes : sortie de dimension `2 x units`. Necessite la sequence complete
(pas de generation autoregressive).

In [ ]:
model_bilstm = build_model('bilstm')
model_bilstm.summary()
bl = next(l for l in model_bilstm.layers if l.__class__.__name__ == 'Bidirectional')
print(f"\nParametres BiLSTM = 2 x LSTM = 2 x {keras_lstm:,} = {bl.count_params():,}")

In [ ]:
history_bilstm = train(model_bilstm, 'bilstm')

---
## Partie 6 - Modele 3 : Stacked BiLSTM

Deux couches BiLSTM : la 1re (`return_sequences=True`) transmet la sequence complete a la 2e.
Couche 1 -> patterns locaux ; couche 2 -> structures plus abstraites.

In [ ]:
model_stacked = build_model('stacked')
model_stacked.summary()

# Dimension d'entree de la 2e couche : la 1re BiLSTM(64) sort 2*64 = 128
d2, n2 = 32, 128
manuel2 = 2 * 4 * d2 * (d2 + n2 + 1)
keras2 = [l for l in model_stacked.layers if l.__class__.__name__ == 'Bidirectional'][1].count_params()
print(f"\n2e couche - manuel : 2 x 4 x {d2} x ({d2}+{n2}+1) = {manuel2:,} | Keras : {keras2:,}")

In [ ]:
history_stacked = train(model_stacked, 'stacked_bilstm')

---
## Partie 7 - Comparaison

In [ ]:
def plot_comparison(histories, names, colors):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    for h, name, c in zip(histories, names, colors):
        a1.plot(h.history['accuracy'], c=c, label=f'{name} (train)')
        a1.plot(h.history['val_accuracy'], c=c, ls='--', label=f'{name} (val)')
        a2.plot(h.history['loss'], c=c, label=f'{name} (train)')
        a2.plot(h.history['val_loss'], c=c, ls='--', label=f'{name} (val)')
    a1.set_title('Accuracy'); a1.set_xlabel('epoch'); a1.legend(fontsize=8); a1.grid(alpha=.3)
    a2.set_title('Loss'); a2.set_xlabel('epoch'); a2.legend(fontsize=8); a2.grid(alpha=.3)
    plt.tight_layout(); plt.show()

histories = [history_lstm, history_bilstm, history_stacked]
names     = ['LSTM Simple', 'BiLSTM', 'Stacked BiLSTM']
colors    = ['#2196F3', '#FF5722', '#4CAF50']
plot_comparison(histories, names, colors)

In [ ]:
models = [('LSTM Simple', model_lstm), ('BiLSTM', model_bilstm), ('Stacked BiLSTM', model_stacked)]
results = {}
for name, m in models:
    loss, acc = m.evaluate(X_test_pad, y_test, verbose=0)
    results[name] = {'loss': loss, 'accuracy': acc}
    print(f"{name:20s} acc={acc:.4f}  loss={loss:.4f}")

plt.figure(figsize=(8, 5))
accs = [results[n]['accuracy'] for n in names]
bars = plt.bar(names, accs, color=colors, edgecolor='white')
for b, a in zip(bars, accs):
    plt.text(b.get_x()+b.get_width()/2, a+0.002, f'{a:.4f}', ha='center', fontweight='bold')
plt.ylim(0.80, 0.92); plt.ylabel('test accuracy'); plt.title('Comparaison - IMDB')
plt.grid(axis='y', alpha=.3); plt.tight_layout(); plt.show()

---
## Partie 8 - Analyse : rapport, matrice de confusion, MC Dropout

On reutilise le **Dropout a l'inference** (`training=True`) pour estimer l'incertitude
et tracer le compromis accuracy / taux de rejet.

In [ ]:
def mc_dropout_predict(model, X, T=50):
    preds = np.array([model(X, training=True).numpy() for _ in range(T)])
    return preds.mean(axis=0), preds.var(axis=0)


def report(model, name, sample_X, sample_y):
    # Rapport + matrice de confusion
    y_pred = (model.predict(X_test_pad, verbose=0) > 0.5).astype(int).flatten()
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=['Negatif', 'Positif']))
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['Neg', 'Pos']).plot(ax=ax, cmap='Blues')
    ax.set_title(name); plt.tight_layout(); plt.show()

    # MC Dropout : incertitude
    mean, var = mc_dropout_predict(model, sample_X)
    mean, var = mean.flatten(), var.flatten()
    cls = (mean > 0.5).astype(int)
    ok = cls == sample_y.flatten()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    a1.hist(var[ok], bins=30, alpha=.6, color='green', label='corrects')
    a1.hist(var[~ok], bins=30, alpha=.6, color='red', label='incorrects')
    a1.set_title('Incertitude (MC Dropout)'); a1.set_xlabel('variance'); a1.legend(); a1.grid(alpha=.3)

    thr = np.linspace(0, var.max(), 50)
    acc = [ (cls[var <= t] == sample_y.flatten()[var <= t]).mean() if (var<=t).sum() else 1.0 for t in thr]
    rej = [ 1 - (var <= t).mean() for t in thr]
    a2b = a2.twinx()
    a2.plot(thr, acc, 'b-', label='accuracy (acceptees)')
    a2b.plot(thr, rej, 'r--', label='taux de rejet')
    a2.set_xlabel('seuil incertitude'); a2.set_ylabel('accuracy', color='b'); a2b.set_ylabel('rejet', color='r')
    a2.set_title('Accuracy vs rejet'); a2.grid(alpha=.3)
    plt.tight_layout(); plt.show()

sample_X, sample_y = X_test_pad[:600], y_test[:600]
for name, m in models:
    report(m, name, sample_X, sample_y)

---
## Partie 9 - Phrases personnalisees

In [ ]:
def predict_sentiment(text, model, word_index, max_len=MAX_LEN, vocab=VOCAB_SIZE):
    enc = []
    for w in text.lower().split():
        idx = word_index.get(w, 0)            # rang du mot (0 si inconnu)
        code = idx + 3                         # offset IMDB (PAD/START/UNK)
        enc.append(code if 3 <= code < vocab else 2)   # 2 = <UNK> si hors vocabulaire
    padded = pad_sequences([enc], maxlen=max_len, padding='pre', truncating='pre')
    score = float(model.predict(padded, verbose=0)[0][0])
    sentiment = 'POSITIF' if score > 0.5 else 'NEGATIF'
    return sentiment, score, (score if score > 0.5 else 1 - score)

phrases = [
    "This movie was absolutely amazing, I loved every minute of it!",
    "Terrible film, worst I have ever seen. Complete waste of time.",
    "The movie was not bad at all, quite enjoyable actually.",
    "It started well but the ending was disappointing and confusing.",
    "A masterpiece of cinema, brilliant acting and stunning visuals.",
    "I fell asleep during this boring and predictable movie.",
]
for p in phrases:
    s, score, conf = predict_sentiment(p, model_bilstm, word_index)
    print(f"[{s}] conf={conf:.0%}  \"{p}\"")

---
## Partie 10 - Recap

In [ ]:
print(f"{'Modele':<22}{'params':>12}{'test acc':>10}{'test loss':>11}")
print('-' * 55)
for name, m in models:
    print(f"{name:<22}{m.count_params():>12,}{results[name]['accuracy']:>10.4f}{results[name]['loss']:>11.4f}")

print("\nObservations attendues :")
print("- BiLSTM > LSTM simple (contexte bidirectionnel), tete identique => effet isole")
print("- Stacked BiLSTM : plus de parametres, pas forcement meilleur (overfitting possible)")

## Exercices
1. Ajouter un `GlobalMaxPooling1D` apres un BiLSTM `return_sequences=True` au lieu du dernier etat.
2. Remplacer l'Embedding appris par des embeddings **GloVe** pre-entraines.
3. Analyser les 10 critiques les plus mal classees par le BiLSTM.
4. Faire varier `EMBEDDING_DIM`, `units`, `MAX_LEN`, `dropout`.

## References
- Hochreiter & Schmidhuber (1997) ; Schuster & Paliwal (1997) ; Maas et al. (2011, IMDB).